In [19]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
# Chat Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if temperature:
        params["temperature"] = temperature

    message = client.messages.create(**params)
    return message.content[0].text

In [21]:
# Function to grade a test case + output using a model
import json

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [22]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
        """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [23]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [24]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each test case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [25]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [26]:
print(json.dumps(results, indent=4))

[
    {
        "output": "# AWS CloudFormation Template Parser\n\nHere's a Python function that parses an AWS CloudFormation template and returns all resource logical IDs:\n\n```python\nimport json\nfrom typing import List\n\ndef get_cloudformation_resource_ids(template: str) -> List[str]:\n    \"\"\"\n    Parse an AWS CloudFormation template (JSON string) and return a list of all resource logical IDs.\n    \n    Args:\n        template (str): A JSON string containing the CloudFormation template\n        \n    Returns:\n        List[str]: A list of resource logical IDs\n        \n    Raises:\n        json.JSONDecodeError: If the template is not valid JSON\n        KeyError: If the template doesn't have a 'Resources' section\n    \"\"\"\n    try:\n        # Parse the JSON template\n        template_dict = json.loads(template)\n        \n        # Extract the Resources section\n        resources = template_dict.get('Resources', {})\n        \n        # Return the list of resource logica